# 01 · Extração dos embeddings FaRL (rodar no Google Colab com GPU)

**O que este notebook faz:** passa cada foto de rosto pelo FaRL congelado e salva um vetor de 512 números por imagem.
Depois disso, **nenhuma etapa do projeto precisa mais de GPU**: todo o treino roda em cima do arquivo `features_farl.parquet`.

**Antes de rodar**
1. `Ambiente de execução > Alterar o tipo de ambiente > GPU T4`
2. No seu Google Drive, crie `MyDrive/projeto-ml/raw/` e coloque:
   - `UTKFace.tar.gz` (versão *Aligned & Cropped*, ~100 MB, ~23,7 mil imagens)
   - `FGNET.zip` (1.002 imagens, nomes no padrão `001A02.JPG`)
3. Ajuste `REPO_URL` abaixo para o seu repositório no GitHub

Tempo esperado na T4: ~10 a 15 min no total.

## 1. Ambiente

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/veridade.git"   # <- ajuste
DRIVE_DIR = "/content/drive/MyDrive/projeto-ml"

from google.colab import drive
drive.mount("/content/drive")

!git clone -q $REPO_URL /content/repo || echo "repo já clonado"
%cd /content/repo
!pip install -q "transformers>=4.40" pyarrow
!pip install -q --no-deps facenet-pytorch   # só o MTCNN, para recortar rostos do FG-NET

import torch, sys
sys.path.insert(0, "/content/repo")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (troque o ambiente!)")

## 2. Descompactar os datasets

In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)
!tar -xzf $DRIVE_DIR/raw/UTKFace.tar.gz -C data/raw/
!unzip -q -o $DRIVE_DIR/raw/FGNET.zip -d data/raw/FGNET
!ls data/raw

## 3. Ler rótulos pelos nomes dos arquivos

In [ ]:
import pandas as pd
from src.config import IDADE_MAX
from src.data.parsers import parse_utkface, parse_fgnet

utk = parse_utkface("data/raw/UTKFace")
fg = parse_fgnet("data/raw/FGNET")
meta = pd.concat([utk, fg], ignore_index=True)

antes = len(meta)
meta = meta[meta["idade"] <= IDADE_MAX].reset_index(drop=True)
print(f"Removidas {antes - len(meta)} imagens com idade > {IDADE_MAX} (fora das 102 classes do artigo)")
meta.groupby("dataset")["idade"].describe()

## 4. Recorte de rosto no FG-NET

O UTKFace já vem alinhado e recortado. O FG-NET vem com a foto inteira, então recortamos o rosto com o MTCNN
(com margem) para ficar parecido com o que o FaRL espera. Se nenhum rosto for detectado, usamos recorte central e marcamos na coluna `rosto_detectado`.

In [ ]:
from PIL import Image
from facenet_pytorch import MTCNN

mtcnn = MTCNN(keep_all=False, device="cuda" if torch.cuda.is_available() else "cpu")

def recortar_rosto(caminho, margem=0.25):
    im = Image.open(caminho).convert("RGB")
    caixas, _ = mtcnn.detect(im)
    if caixas is None:
        w, h = im.size; s = min(w, h)
        return im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)), False
    x1, y1, x2, y2 = caixas[0]
    mw, mh = (x2 - x1) * margem, (y2 - y1) * margem
    return im.crop((max(0, x1 - mw), max(0, y1 - mh), x2 + mw, y2 + mh)), True

def carregar(linha):
    if linha.dataset == "FG-NET":
        return recortar_rosto(linha.arquivo)
    return Image.open(linha.arquivo).convert("RGB"), True

## 5. Extrair os embeddings

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from src.config import EMB_COLS
from src.data.farl import FarlExtractor

ext = FarlExtractor()
LOTE = 128
embs, detectado = [], []
for i in tqdm(range(0, len(meta), LOTE)):
    ims, flags = zip(*(carregar(l) for l in meta.iloc[i:i + LOTE].itertuples()))
    embs.append(ext.embed(list(ims)))
    detectado.extend(flags)

E = np.vstack(embs)
meta["rosto_detectado"] = detectado
print(E.shape, "| normas (devem ser ~1):", np.linalg.norm(E, axis=1)[:5].round(3))
print("FG-NET sem rosto detectado:", (~meta.loc[meta.dataset == 'FG-NET', 'rosto_detectado']).sum())

## 6. Splits congelados e salvamento

In [ ]:
from src.data.splits import criar_splits

feats = pd.concat([meta, pd.DataFrame(E, columns=EMB_COLS)], axis=1)
feats["arquivo"] = feats["arquivo"].str.replace("/content/repo/", "", regex=False)
feats = criar_splits(feats)
print(pd.crosstab(feats["dataset"], feats["split"]))

os.makedirs("data/processed", exist_ok=True)
feats.to_parquet("data/processed/features_farl.parquet", index=False)
!mkdir -p $DRIVE_DIR/processed && cp data/processed/features_farl.parquet $DRIVE_DIR/processed/
print("Salvo no Drive:", f"{DRIVE_DIR}/processed/features_farl.parquet")

## 7. Teste de sanidade

Uma regressão Ridge simples sobre os embeddings. Se o MAE no `val` ficar na casa de **4 a 6 anos**, os embeddings estão bons.
Se passar de 10, algo deu errado (imagem errada, normalização, rótulo).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

tr, va, fx = (feats[feats.split == s] for s in ["train", "val", "test_externo"])
ridge = Ridge(alpha=1.0).fit(tr[EMB_COLS], tr["idade"])
print(f"MAE val (UTKFace): {mean_absolute_error(va['idade'], ridge.predict(va[EMB_COLS])):.2f} anos")
print(f"MAE FG-NET (externo): {mean_absolute_error(fx['idade'], ridge.predict(fx[EMB_COLS])):.2f} anos")

**Pronto.** Baixe `features_farl.parquet` do Drive e coloque em `data/processed/` no seu repositório local.
Esse arquivo **não vai para o GitHub** (está no `.gitignore`): o README explica como gerar.